# Combine dataset with pandas
This demonstrate how to match rows in different dataframes

In [10]:
import pandas as pd
import random
import string
import time
from collections import defaultdict
from pathlib import Path
from tqdm.auto import tqdm

### Creat sample data

In [66]:
# Number of samples
n = 25000

# Function to generate random strings
def random_string(length=8):
    return ''.join(random.choices(string.ascii_lowercase, k=length))

# Generate random document names
random_docs = [random_string() for _ in range(n)]
random_text = [random_string() for _ in range(n)]

# Create df1 using random document names
filenames = [f"{doc}_pagenr_{j}.csv" for doc in random_docs for j in range(2)]
pagenrs = [j for _ in random_docs for j in range(2)]
df1 = pd.DataFrame({
    "filename": filenames,
    "pagenr": pagenrs
})

# Create df2 from unique document names
df2 = pd.DataFrame({
    "document": random_docs,
    "text": random_text
})

print(len(df1))
print(len(df2))

50000
25000


In [67]:
df1.head()

,filename,pagenr
0,jzkbfgkf_pagenr_0.csv,0
1,jzkbfgkf_pagenr_1.csv,1
2,niozueup_pagenr_0.csv,0
3,niozueup_pagenr_1.csv,1
4,fwivfqje_pagenr_0.csv,0


In [68]:
df2.head()

,document,text
0,jzkbfgkf,tirgrkag
1,niozueup,blatrxba
2,fwivfqje,sijnkarc
3,mtkskkpf,bluxoxdw
4,hvofzevi,zzujsfls


### Combine dataset

In [70]:
# mask work like this
mask = df1["filename"].str.contains("jzkbfgkf", na=False)
df1[mask]

,filename,pagenr
0,jzkbfgkf_pagenr_0.csv,0
1,jzkbfgkf_pagenr_1.csv,1


In [79]:
match_indices = df1.index[mask]
match_indices

Index([0, 1], dtype='int64')

### Example 1 with iterrows

In [72]:
def update_text_column_rows(df1, df2):
    """
    Updates the 'text' column in df1 based on matching 'filename' and 'document_name' from df2.

    Parameters:
    - df1: DataFrame with a 'filename' column and a 'text' column to update.
    - df2: DataFrame with 'document_name' and 'text' columns.

    Returns:
    - Updated df1 DataFrame.
    """
    df1_copy = df1.copy(deep=True)
    start_time = time.time()

    df1_copy["text"] = None  # Initialize the column

    for idx, row in df2.iterrows():
        doc_name = row["document"]
        text_value = row["text"]

        # Find matching row in df1
        mask = df1_copy["filename"].astype(str).str.contains(doc_name, na=False)
        match_indices = df1_copy.index[mask]

        if len(match_indices) == 0:
            print(f"Warning: No match found for '{doc_name}'")
        else:
            for idx in match_indices:
                df1.at[idx, "text"] = text_value

    elapsed_time = time.time() - start_time
    print(f"Function completed in {elapsed_time:.4f} seconds")

    return df1_copy


In [73]:
dfr = update_text_column_rows(df1, df2)

Function completed in 409.9574 seconds


### Example 2 with itertuples

In [74]:
def update_text_column_tuples(df1, df2):
    """
    Updates the 'text' column in df1 based on matching 'filename' and 'document_name' from df2.

    Parameters:
    - df1: DataFrame with a 'filename' column and a 'text' column to update.
    - df2: DataFrame with 'document_name' and 'text' columns.

    Returns:
    - Updated df1 DataFrame.
    """
    df1_copy = df1.copy(deep=True)
    start_time = time.time()

    df1_copy["text"] = None  # Initialize the column

    for row in df2.itertuples(index=False):
        doc_name = row["document"]
        text_value = row["text"]

        # Find matching row in df1
        mask = df1_copy["filename"].astype(str).str.contains(doc_name, na=False)
        match_indices = df1_copy.index[mask]

        if len(match_indices) == 0:
            print(f"Warning: No match found for '{doc_name}'")
        else:
            for idx in match_indices:
                df1.at[idx, "text"] = text_value

    elapsed_time = time.time() - start_time
    print(f"Function completed in {elapsed_time:.4f} seconds")

    return df1_copy

In [75]:
dft = update_text_column_rows(df1, df2)

Function completed in 412.1236 seconds


### Example 3 with loc

In [76]:
def update_text_column_loc(df1, df2):
    """
    Updates the 'text' column in df1 based on matching 'filename' and 'document_name' from df2.

    Parameters:
    - df1: DataFrame with a 'filename' column and a 'text' column to update.
    - df2: DataFrame with 'document_name' and 'text' columns.

    Returns:
    - Updated df1 DataFrame.
    """
    df1_copy = df1.copy(deep=True)
    start_time = time.time()

    df1_copy["text"] = None  # Initialize the column

    for idx, row in df2.iterrows():
        doc_name = row["document"]
        text_value = row["text"]

        # Find matching row in df1
        mask = df1_copy["filename"].astype(str).str.contains(doc_name, na=False)

        match_count = mask.sum()
        if match_count == 0:
            print(f"Warning: No match found for '{doc_name}'")
        else:
            df1_copy.loc[mask, "text"] = text_value

    elapsed_time = time.time() - start_time
    print(f"Function completed in {elapsed_time:.4f} seconds")

    return df1_copy


In [77]:
dfl = update_text_column_loc(df1, df2)

Function completed in 419.0463 seconds
